In [0]:
-- =====================================================================================
-- Notebook: 00_admin/00_uc_setup.sql
-- Project : nba-datalakehouse
-- Purpose : One-time Unity Catalog (UC) setup for this project:
--           1) Confirm which catalog(s) you have access to (Free Edition is UC-first)
--           2) Select the catalog we will use for all project assets
--           3) Create project schemas (databases) for Bronze/Silver layers
--           4) (Optional) Create a schema to hold Volumes (file storage in UC)
--           5) Run a small smoke test to confirm you can create Delta tables
--
-- IMPORTANT CONCEPTS
-- ------------------
-- Workspace folders (left sidebar) are for *code artifacts* (notebooks, files, repos).
-- Unity Catalog is for *data assets* (catalogs, schemas, tables, views, volumes).
--
-- In your environment:
--   - Hive Metastore is disabled (legacy mode is off).
--   - You saw catalogs: samples, system, workspace.
--   - 'workspace' is the catalog where you can create your own schemas/tables.
--
-- Naming convention we will follow everywhere:
--   workspace.<schema>.<table>
-- Examples:
--   workspace.nba_bronze.balldontlie_games
--   workspace.nba_silver.fact_game
--   workspace.nba_files.landing   (Volume)
--
-- Why separate schemas?
--   - Bronze: raw ingested data (minimal changes, mostly schema-on-read)
--   - Silver: cleaned/typed/flattened data ready for analytics & Gold later
--   - Files: UC Volumes for landing/checkpoints/exports (storage layer)
-- =====================================================================================


-- =====================================================================================
-- 1) Inspect current session context
--    This helps you understand what catalog/schema your notebook is currently "in".
--    If you ever get errors like "object not found", this is the first thing to check.
-- =====================================================================================
SELECT
  current_catalog() AS current_catalog,
  current_schema()  AS current_schema;


-- =====================================================================================
-- 2) List catalogs you can use
--    In Free Edition UC-first workspaces you often see:
--      - system  : internal
--      - samples : demo datasets
--      - workspace : your writable project catalog
-- =====================================================================================
SHOW CATALOGS;


-- =====================================================================================
-- 3) Choose the catalog for this project
--    We will standardize on: workspace
--    This avoids confusion and makes all object names consistent.
-- =====================================================================================
USE CATALOG workspace;


-- =====================================================================================
-- 4) Create schemas (aka databases) for medallion layers
--    These schemas are where your Delta tables will live.
--    NOTE: Schemas appear in Catalog Explorer under the 'workspace' catalog.
-- =====================================================================================

CREATE SCHEMA IF NOT EXISTS nba_bronze
COMMENT 'Bronze layer: raw ingested data stored as Delta tables (minimal transforms).';

CREATE SCHEMA IF NOT EXISTS nba_silver
COMMENT 'Silver layer: cleaned/typed/flattened Delta tables built from Bronze.';


-- =====================================================================================
-- 5) Create a schema for UC Volumes (storage)
--    Why do we need this?
--      - Public DBFS root is disabled in your workspace
--      - Reading file:/Workspace/... was blocked by policy
--      - UC Volumes provide a governed storage path Spark can read/write:
--          dbfs:/Volumes/<catalog>/<schema>/<volume>/
--
--    We'll store landing files here (raw JSONL), and later can add:
--      - checkpoints (for streaming in v2)
--      - exports (for sharing outputs)
-- =====================================================================================
CREATE SCHEMA IF NOT EXISTS nba_files
COMMENT 'Storage schema for UC Volumes (landing/checkpoints/exports) for NBA project.';


-- =====================================================================================
-- 6) Verify schemas were created
-- =====================================================================================
SHOW SCHEMAS IN workspace;


-- =====================================================================================
-- 7) (Optional) Set a default schema for this notebook session
--    This makes unqualified table names resolve here.
--    Even so, best practice in shared notebooks is to use fully-qualified names.
-- =====================================================================================
USE SCHEMA nba_bronze;


-- =====================================================================================
-- 8) Smoke test: prove we can create and drop a Delta table in UC
--    Why do this?
--      - It validates permissions and that Delta works in your chosen catalog/schema.
--      - If this fails, nothing else will work and we should fix it now.
-- =====================================================================================
CREATE TABLE IF NOT EXISTS _smoke_test_table (
  id INT COMMENT 'Temporary smoke test column'
)
USING DELTA
COMMENT 'Temporary table used to confirm UC + Delta table creation works.';

-- Confirm it exists
SHOW TABLES IN workspace.nba_bronze;

-- Clean up
DROP TABLE _smoke_test_table;


-- =====================================================================================
-- End of notebook.
-- Next notebook (recommended):
--   00_admin/01_volumes_setup.sql
-- which will create volumes like:
--   workspace.nba_files.landing
-- and (optionally) workspace.nba_files.checkpoints
-- =====================================================================================

